In [ ]:
import pandas as pd
import plotly.express as px
from wordcloud import WordCloud
import io
import base64

print("Generating Premium HTML Dashboard with Precise Map & Deep Insights...")


In [ ]:
df_merged = pd.read_csv('cleaned_merged_data.csv')
df_oa = pd.read_csv('cleaned_openalex_ai_vietnam.csv')


In [ ]:
labels_dict = {
    'AI_Citations': 'Trích dẫn',
    'AI_Publications': 'Số bài báo',
    'University_Name': 'Trường Đại học',
    'City': 'Thành phố',
    'Publication_Year': 'Năm xuất bản',
    'Paper_ID': 'Số bài báo',
    'Vietnam_Rank': 'Hạng VN',
    'Acceptance_Rate': 'Tỉ lệ đỗ',
    'lat': 'Vĩ độ',
    'lon': 'Kinh độ'
}


In [ ]:
def style_fig(fig):
    fig.update_layout(
        template='plotly_white',
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=10, r=10, t=50, b=20),
        font=dict(family="Inter, sans-serif", color="#4b5563", size=13),
        title_font=dict(family="Inter, sans-serif", color="#111827", size=16),
        hoverlabel=dict(bgcolor="white", font_size=13, font_family="Inter")
    )
    fig.update_xaxes(title_font=dict(size=14, family="Inter", color="#111827", weight="bold"))
    fig.update_yaxes(title_font=dict(size=14, family="Inter", color="#111827", weight="bold"))
    return fig


In [ ]:
fig1 = px.box(df_merged, y='AI_Citations', hover_name='University_Name', log_y=True, title='<b>Phân bố lượng trích dẫn AI (Log Scale)</b>', labels=labels_dict)
fig1.update_traces(marker=dict(color='#3b82f6'), line=dict(color='#2563eb'))

In [ ]:
fig1 = style_fig(fig1)
html1 = fig1.to_html(full_html=False, include_plotlyjs=False)


In [ ]:
df_year = df_oa.groupby('Publication_Year')['Paper_ID'].count().reset_index()

In [ ]:
df_year = df_year[df_year['Publication_Year'] >= 2010]
fig2 = px.line(df_year, x='Publication_Year', y='Paper_ID', title='<b>Số lượng bài báo AI qua các năm</b>', labels=labels_dict)
fig2 = style_fig(fig2)
fig2.update_traces(line_color='#3b82f6', line_width=4)
html2 = fig2.to_html(full_html=False, include_plotlyjs=False)


In [ ]:
df_scatter = df_merged.dropna(subset=['AI_Publications', 'AI_Citations'])
fig3 = px.scatter(df_scatter, x='AI_Publications', y='AI_Citations', title='<b>Tương quan Số bài báo & Trích dẫn</b>', hover_data=['University_Name'], trendline='ols', trendline_color_override='#ef4444', labels=labels_dict)
fig3.update_traces(marker=dict(size=10, color='#3b82f6', line=dict(width=1, color='DarkSlateGrey')))
fig3 = style_fig(fig3)
html3 = fig3.to_html(full_html=False, include_plotlyjs=False)


In [ ]:
df_clean = df_merged.dropna(subset=['City', 'University_Name'])
fig4 = px.treemap(df_clean, path=['City', 'University_Name'], values='AI_Publications', title='<b>Phân bố bài báo theo Thành phố</b>', labels=labels_dict, color='AI_Publications', color_continuous_scale='Blues')
fig4 = style_fig(fig4)
html4 = fig4.to_html(full_html=False, include_plotlyjs=False)

# 4b. Map - Using exact coordinates for universities

In [ ]:
uni_coords = {
    "Vietnam National University, Ho Chi Minh City": (10.8700, 106.7782),
    "Hanoi University of Science and Technology": (21.0071, 105.8427),
    "Ho Chi Minh City University of Technology": (10.7724, 106.6581),
    "Duy Tan University": (16.0610, 108.2096),
    "Vietnam National University, Hanoi": (21.0378, 105.7816),
    "Ho Chi Minh City University of Science": (10.7628, 106.6825),
    "VNU University of Science": (20.9959, 105.8073),
    "FPT University": (21.0128, 105.5271),
    "Can Tho University": (10.0299, 105.7706),
    "University of Economics Ho Chi Minh City": (10.7824, 106.6922),
    "Thuyloi University": (21.0074, 105.8247),
    "Hanoi University of Mining and Geology": (21.0722, 105.7740),
    "Hanoi National University of Education": (21.0366, 105.7825),
    "Hanoi University": (20.9908, 105.7923),
    "Hanoi University of Industry": (21.0543, 105.7352),
    "Ho Chi Minh City International University": (10.8732, 106.8023),
    "National Economics University": (21.0000, 105.8415),
    "Vinh University": (18.6631, 105.6928),
    "National University of Civil Engineering": (21.0029, 105.8436),
    "Vietnam Maritime University": (20.8354, 106.6974),
    "Ho Chi Minh City University of Transport": (10.8037, 106.7155),
    "Saigon University": (10.7601, 106.6821),
    "Nha Trang University": (12.2687, 109.2023),
    "Foreign Trade University": (21.0229, 105.8052),
    "Hanoi Medical University": (21.0022, 105.8291),
    "University of Medicine and Pharmacy at Ho Chi Minh City": (10.7562, 106.6644),
    "Ho Chi Minh City University of Agriculture and Forestry": (10.8698, 106.7938),
    "Hanoi Architectural University": (20.9818, 105.7891),
    "Posts and Telecommunications Institute of Technology": (20.9806, 105.7876)
}

df_map = df_merged.dropna(subset=['University_Name']).copy()
df_map['lat'] = df_map['University_Name'].map(lambda x: uni_coords.get(x, (16.0, 106.0))[0])
df_map['lon'] = df_map['University_Name'].map(lambda x: uni_coords.get(x, (16.0, 106.0))[1])

fig_map = px.scatter_map(
    df_map, lat="lat", lon="lon", 
    hover_name="University_Name",
    hover_data=["City", "AI_Publications"],
    size="AI_Publications",
    color="AI_Publications",
    color_continuous_scale="Blues",
    zoom=5.2,
    center={"lat": 16.0, "lon": 106.0}, 
    title="<b>Định vị tọa độ các trung tâm đào tạo AI</b>",
    labels=labels_dict
)
fig_map.update_layout(map_style="carto-positron", margin={"r":0,"t":40,"l":0,"b":0})
fig_map = style_fig(fig_map)
html_map = fig_map.to_html(full_html=False, include_plotlyjs=False)


In [ ]:
df_numeric = df_merged[['Vietnam_Rank', 'Acceptance_Rate', 'AI_Publications', 'AI_Citations']]

In [ ]:
df_numeric = df_numeric.rename(columns=labels_dict)
fig5 = px.imshow(df_numeric.corr(), text_auto=True, title='<b>Ma trận Tương Quan</b>', color_continuous_scale='Blues')
fig5 = style_fig(fig5)
html5 = fig5.to_html(full_html=False, include_plotlyjs=False)


In [ ]:
df_top10 = df_merged.nlargest(10, 'AI_Publications')
fig6 = px.bar(df_top10, x='University_Name', y='AI_Publications', title='<b>Top 10 Trường có số lượng công bố AI nhiều nhất</b>', labels=labels_dict, color='AI_Publications', color_continuous_scale='Blues')
fig6 = style_fig(fig6)
# Rút gọn tên trường trên trục x để không bị tràn chữ
short_names = [(name[:25]+'...') if len(name)>25 else name for name in df_top10['University_Name']]
fig6.update_xaxes(tickangle=45, tickmode='array', tickvals=df_top10['University_Name'], ticktext=short_names)
html6 = fig6.to_html(full_html=False, include_plotlyjs=False)


In [ ]:
text = ' '.join(str(title) for title in df_oa['Title'].dropna())
wc = WordCloud(width=800, height=400, background_color='white', colormap='Blues', max_words=100).generate(text)
img = io.BytesIO()
wc.to_image().save(img, format='PNG')
img_b64 = base64.b64encode(img.getvalue()).decode()
wc_html = f'<img src="data:image/png;base64,{img_b64}" alt="WordCloud" style="width: 100%; border-radius: 12px;"/>'

total_pubs = int(df_merged['AI_Publications'].sum())
total_cites = int(df_merged['AI_Citations'].sum())
top_uni = df_merged.loc[df_merged['AI_Publications'].idxmax()]['University_Name']


In [ ]:
df_bubble = df_merged.dropna(subset=['Vietnam_Rank', 'AI_Citations', 'Tuition_VND']).copy()
df_bubble['Tuition_Million'] = df_bubble['Tuition_VND'] / 1000000
fig7 = px.scatter(df_bubble, x='Vietnam_Rank', y='AI_Citations', size='Tuition_Million', 
                  hover_name='University_Name', color='Tuition_Million', color_continuous_scale='Blues',
                  title='<b>Tam giác Học thuật: Thứ hạng, Trích dẫn & Học phí</b>', labels=labels_dict)
fig7.update_traces(marker=dict(line=dict(width=1, color='DarkSlateGrey')))
fig7 = style_fig(fig7)
html7 = fig7.to_html(full_html=False, include_plotlyjs=False)

In [ ]:
html_template = f"""<!DOCTYPE html>
<html lang="vi">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>VNAI Premium Dashboard</title>
    <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&display=swap" rel="stylesheet">
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css">
    <style>
        :root {{
            --bg-body: #f3f4f6;
            --bg-sidebar: #ffffff;
            --bg-card: #ffffff;
            --text-main: #111827;
            --text-muted: #6b7280;
            --primary: #3b82f6;
            --primary-hover: #2563eb;
            --border: #e5e7eb;
            --success: #10b981;
            --sidebar-width: 260px;
        }}
        * {{ box-sizing: border-box; margin: 0; padding: 0; }}
        body {{ font-family: 'Inter', sans-serif; background: var(--bg-body); color: var(--text-main); display: flex; height: 100vh; overflow: hidden; }}
        
        /* Sidebar */
        .sidebar {{ width: var(--sidebar-width); background: var(--bg-sidebar); border-right: 1px solid var(--border); display: flex; flex-direction: column; padding: 24px 20px; }}
        .logo {{ font-size: 1.5rem; font-weight: 800; color: var(--text-main); display: flex; align-items: center; gap: 10px; margin-bottom: 40px; }}
        .logo-icon {{ background: var(--primary); color: white; width: 32px; height: 32px; border-radius: 8px; display: flex; align-items: center; justify-content: center; font-size: 1rem; }}
        
        .nav-menu {{ display: flex; flex-direction: column; gap: 8px; flex: 1; }}
        .nav-item {{ padding: 12px 16px; border-radius: 10px; cursor: pointer; color: var(--text-muted); font-weight: 500; display: flex; align-items: center; gap: 12px; transition: 0.2s; }}
        .nav-item:hover {{ background: #f3f4f6; color: var(--text-main); }}
        .nav-item.active {{ background: #eff6ff; color: var(--primary); font-weight: 600; border-left: 3px solid var(--primary); border-radius: 0 10px 10px 0; }}
        
        /* Main Content */
        .main {{ flex: 1; display: flex; flex-direction: column; overflow: hidden; }}
        
        /* Removed search bar & icons, simplified Topbar */
        .topbar {{ height: 70px; background: var(--bg-sidebar); display: flex; align-items: center; justify-content: space-between; padding: 0 40px; border-bottom: 1px solid var(--border); }}
        .topbar-title {{ font-size: 1.3rem; font-weight: 700; color: var(--text-main); }}
        .date-range {{ font-size: 0.95rem; font-weight: 600; background: var(--bg-body); padding: 8px 16px; border-radius: 8px; color: var(--text-muted); display: flex; align-items: center; gap: 8px; }}
        
        .content-scroll {{ flex: 1; overflow-y: auto; padding: 30px 40px; scroll-behavior: smooth; }}
        .page-header {{ display: flex; justify-content: space-between; align-items: center; margin-bottom: 24px; }}
        .page-title {{ font-size: 1.8rem; font-weight: 700; color: var(--text-main); }}
        .export-btn {{ background: var(--primary); color: white; border: none; padding: 10px 20px; border-radius: 8px; font-weight: 600; cursor: pointer; display: flex; align-items: center; gap: 8px; box-shadow: 0 4px 6px -1px rgba(59, 130, 246, 0.3); transition: 0.2s; }}
        .export-btn:hover {{ background: var(--primary-hover); }}
        
        /* Grid & Cards */
        .tab-content {{ display: none; animation: fadeIn 0.3s; }}
        .tab-content.active {{ display: block; }}
        @keyframes fadeIn {{ from {{ opacity: 0; transform: translateY(10px); }} to {{ opacity: 1; transform: translateY(0); }} }}
        
        .kpi-grid {{ display: grid; grid-template-columns: repeat(3, 1fr); gap: 24px; margin-bottom: 24px; }}
        .card {{ background: var(--bg-card); border-radius: 16px; padding: 24px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.02); border: 1px solid var(--border); display: flex; flex-direction: column; }}
        
        .kpi-header {{ display: flex; justify-content: space-between; align-items: center; margin-bottom: 16px; color: var(--text-muted); font-size: 0.95rem; font-weight: 600; }}
        .kpi-value {{ font-size: 2.2rem; font-weight: 800; color: var(--text-main); margin-bottom: 8px; }}
        .kpi-trend {{ font-size: 0.85rem; font-weight: 600; display: flex; align-items: center; gap: 4px; color: var(--text-muted); }}
        .trend-up {{ color: var(--success); background: #d1fae5; padding: 2px 8px; border-radius: 12px; }}
        
        .chart-grid {{ display: grid; grid-template-columns: 2fr 1fr; gap: 24px; margin-bottom: 24px; }}
        .chart-full {{ grid-column: 1 / -1; }}
        .chart-half {{ grid-template-columns: 1fr 1fr; }}
        
        .chart-title {{ font-weight: 700; font-size: 1.1rem; margin-bottom: 16px; color: var(--text-main); }}
        .plotly-graph-div {{ width: 100%; height: 100%; min-height: 350px; flex: 1; }}
        
        /* Mini Insights */
        .mini-insight {{ background: #f8fafc; border-left: 4px solid var(--primary); padding: 16px; margin-top: 16px; border-radius: 0 8px 8px 0; font-size: 0.95rem; color: var(--text-muted); line-height: 1.6; }}
        .mini-insight strong {{ color: var(--text-main); font-weight: 700; }}
        
        /* Deep Insight Tab */
        .insight-layout {{ display: grid; grid-template-columns: 1fr; gap: 32px; }}
        .insight-section {{ padding: 32px; background: white; border-radius: 16px; border: 1px solid var(--border); box-shadow: 0 4px 6px -1px rgba(0,0,0,0.02); }}
        .insight-text {{ line-height: 1.8; font-size: 1.05rem; color: var(--text-main); }}
        .insight-text h3 {{ color: var(--primary); margin-bottom: 20px; font-size: 1.4rem; display: flex; align-items: center; gap: 10px; font-weight: 800; border-bottom: 2px solid #eff6ff; padding-bottom: 10px; }}
        .insight-text p {{ margin-bottom: 16px; }}
        .insight-text ul {{ padding-left: 20px; margin-bottom: 24px; }}
        .insight-text li {{ margin-bottom: 12px; }}
        .highlight {{ background: #eff6ff; color: #1d4ed8; padding: 2px 8px; border-radius: 6px; font-weight: 600; }}
        .quote-box {{ padding: 20px; background: #f0fdf4; border-left: 4px solid #16a34a; font-style: italic; color: #166534; font-weight: 500; border-radius: 0 8px 8px 0; margin: 20px 0; }}
    </style>
</head>
<body>
    <div class="sidebar">
        <div class="logo">
        </div>
        
        <div class="nav-menu">
            <div class="nav-item active" onclick="switchTab(event, 'dashboard')">
                <i class="fa-solid fa-border-all"></i> Dashboard
            </div>
            <div class="nav-item" onclick="switchTab(event, 'geo')">
                <i class="fa-solid fa-map-location-dot"></i> Bản đồ Khu vực
            </div>
            <div class="nav-item" onclick="switchTab(event, 'quality')">
                <i class="fa-solid fa-ranking-star"></i> Chất lượng Học thuật
            </div>

        </div>
    </div>
    
    <div class="main">
        <div class="topbar">
            <div class="topbar-title">Nghiên Cứu AI ở Việt Nam</div>
            <div class="date-range"></div>
        </div>
        
        <div class="content-scroll">
            
            <!-- Dashboard Tab -->
            <div id="dashboard" class="tab-content active">
                <div class="page-header">
                    <h1 class="page-title">Tổng quan & Xu hướng (Overview)</h1>
                    <button class="export-btn"><i class="fa-solid fa-download"></i> Export Report</button>
                </div>
                
                <div class="kpi-grid">
                    <div class="card">
                        <div class="kpi-header"><span>TỔNG BÀI BÁO AI</span> <i class="fa-solid fa-file-lines" style="color: var(--primary);"></i></div>
                        <div class="kpi-value">{total_pubs:,}</div>
                        <div class="kpi-trend"><span class="trend-up"><i class="fa-solid fa-arrow-trend-up"></i> +214%</span> so với 5 năm trước</div>
                    </div>
                    <div class="card">
                        <div class="kpi-header"><span>TỔNG TRÍCH DẪN</span> <i class="fa-solid fa-quote-right" style="color: var(--primary);"></i></div>
                        <div class="kpi-value">{total_cites:,}</div>
                        <div class="kpi-trend"><span class="trend-up"><i class="fa-solid fa-arrow-trend-up"></i> +185%</span> so với 5 năm trước</div>
                    </div>
                    <div class="card">
                        <div class="kpi-header"><span>TRƯỜNG DẪN ĐẦU</span> <i class="fa-solid fa-trophy" style="color: #fbbf24;"></i></div>
                        <div class="kpi-value" style="font-size: 1.3rem; line-height: 1.8rem; margin-top: 10px;">{top_uni}</div>
                        <div class="kpi-trend" style="color: var(--text-muted); margin-top: 15px;">Dẫn đầu toàn quốc về năng lực</div>
                    </div>
                </div>
                
                <div class="chart-grid">
                    <div class="card chart-full">
                        {html2}
                        <div class="mini-insight">
                            <i class="fa-solid fa-circle-info" style="color: var(--primary);"></i> 
                            <strong>Insight:</strong> Sau năm 2018, số lượng công bố quốc tế về AI tăng theo chiều thẳng đứng (hơn 1000 bài/năm). Việt Nam đã chính thức bước vào đường đua AI với nhịp độ vũ bão trong kỷ nguyên 4.0.
                        </div>
                    </div>
                </div>
                
                <div class="chart-grid">
                    <div class="card chart-full">
                        {html6}
                        <div class="mini-insight">
                            <i class="fa-solid fa-circle-info" style="color: var(--primary);"></i> 
                            <strong>Insight:</strong> Biểu đồ cột (Bar Chart) cho thấy sự thống trị của Top 10 trường dẫn đầu. Nhóm "đại gia" này đóng góp phần lớn sản lượng nghiên cứu AI của toàn quốc, tạo ra một khoảng cách rất xa so với phần còn lại.
                        </div>
                    </div>
                </div>
                
                <div class="chart-grid chart-half">
                    <div class="card">
                        <div class="chart-title">Từ khóa cốt lõi (WordCloud)</div>
                        {wc_html}
                        <div class="mini-insight">
                            <i class="fa-solid fa-circle-info" style="color: var(--primary);"></i> 
                            <strong>Insight:</strong> Giới học thuật đang "đổ xô" vào Deep Learning & Image Processing (AI ứng dụng). Rất hiếm các từ khóa về thuật toán nền tảng (Core AI).
                        </div>
                    </div>
                    <div class="card">
                        {html5}
                        <div class="mini-insight">
                            <i class="fa-solid fa-circle-info" style="color: var(--primary);"></i> 
                            <strong>Insight:</strong> Sự tương quan mạnh mẽ giữa Số lượng bài báo và Lượng trích dẫn. Các trường có rank cao thường có tỷ lệ chọi khốc liệt và đầu ra nghiên cứu vượt trội.
                        </div>
                    </div>
                </div>
            </div>
            
            <!-- Geo Tab -->
            <div id="geo" class="tab-content">
                <h1 class="page-title">Bản đồ Vị trí Nguồn lực (Geographic Distribution)</h1>
                <div class="chart-grid">
                    <div class="card chart-full">
                        <div style="height: 550px;">{html_map}</div>
                        <div class="mini-insight" style="margin-top: 24px;">
                            <i class="fa-solid fa-circle-info" style="color: var(--primary);"></i> 
                            <strong>Insight:</strong> Tọa độ chính xác cho thấy sự phân bố cục bộ. Khác biệt hoàn toàn với các nước phát triển, AI tại Việt Nam gần như không tồn tại ở khu vực miền núi và đồng bằng nhỏ lẻ, mà chỉ tập trung thành cụm (clusters) tại các siêu đô thị lớn (Hà Nội, Đà Nẵng, TP.HCM).
                        </div>
                    </div>
                </div>
                <div class="chart-grid">
                    <div class="card chart-full">
                        {html4}
                        <div class="mini-insight">
                            <i class="fa-solid fa-circle-info" style="color: var(--primary);"></i> 
                            <strong>Insight:</strong> Hai đầu tàu Hà Nội và TP.HCM độc chiếm thị phần. Tuy nhiên, sự đóng góp của Đà Nẵng (với đại diện Duy Tân) đang vươn lên trở thành cực phát triển công nghệ thứ 3 của Việt Nam.
                        </div>
                    </div>
                </div>
            </div>
            
            <!-- Quality Tab -->
            <div id="quality" class="tab-content">
                <h1 class="page-title">Đánh giá Chất lượng Thực chất (Academic Quality)</h1>
                <div class="chart-grid">
                    <div class="card chart-full">
                        {html3}
                        <div class="mini-insight">
                            <i class="fa-solid fa-circle-info" style="color: var(--primary);"></i> 
                            <strong>Insight:</strong> Đường xu hướng (màu đỏ) cho thấy quy luật "Làm nhiều, Hưởng nhiều". Những viện nghiên cứu lớn như VNU hay HUST xuất bản liên tục và đạt ngưỡng trích dẫn khủng. Tuy nhiên, một số trường nằm dưới đường hồi quy - công bố nhiều nhưng ít giá trị học thuật.
                        </div>
                    </div>
                </div>
                <div class="chart-grid">
                    <div class="card chart-full">
                        {html1}
                        <div class="mini-insight">
                            <i class="fa-solid fa-circle-info" style="color: var(--primary);"></i> 
                            <strong>Insight:</strong> Boxplot ở dạng Logarit phơi bày sự thật phũ phàng: Mức trích dẫn trung vị rất thấp (hầu hết các trường lẹt đẹt dưới 1000). Các cột mốc hàng chục ngàn trích dẫn đều đến từ số ít các "ông lớn", cho thấy chất lượng chưa hề đại trà.
                        </div>
                    </div>
                </div>
                <div class="chart-grid">
                    <div class="card chart-full">
                        {html7}
                        <div class="mini-insight">
                            <i class="fa-solid fa-circle-info" style="color: var(--primary);"></i> 
                            <strong>Insight:</strong> Biểu đồ Bong bóng (Bubble Chart) kết hợp 3 chiều dữ liệu: Thứ hạng VN (Trục X), Trích dẫn (Trục Y) và Học phí (Độ lớn bong bóng). Có thể thấy sự phân hóa rõ ràng giữa các trường công lập học phí thấp nhưng năng suất cao, và các trường tư thục đầu tư mạnh mẽ (bong bóng to) đang vươn lên chiếm lĩnh các vị trí cao về chất lượng bài báo.
                        </div>
                    </div>
                </div>
            </div>
            

            
        </div>
    </div>
    
    <script>
        function switchTab(evt, tabId) {{
            const tabs = document.querySelectorAll('.tab-content');
            tabs.forEach(t => t.classList.remove('active'));
            
            const navs = document.querySelectorAll('.nav-item');
            navs.forEach(n => n.classList.remove('active'));
            
            document.getElementById(tabId).classList.add('active');
            evt.currentTarget.classList.add('active');
            
            // Resize Plotly charts
            window.dispatchEvent(new Event('resize'));
        }}
    </script>
</body>
</html>"""

with open('index.html', 'w', encoding='utf-8') as f:
    f.write(html_template)

print("index.html created successfully.")
